# Gaussian Discriminant Analysis

When we have a classfication problem in which the input features $x$ are continous-valued random variables, we can then use the GDA model, which models $p(x|y)$ using a multivariate normal distribution. The model is: $$\begin{align*} y & \sim & \text{Bernoulli}(\phi) \\ x|y=0 & \sim & \mathcal{N}(\mu _0,\Sigma) \\ x|y=1 & \sim &\mathcal{N}(\mu _1,\Sigma)\end{align*}$$

Writing out the distribution, this is: $$\begin{align*} p(y) & = & \phi ^y (1-\phi)^{1-y} \\ p(x|y=0) & = & \frac{1}{(2\pi)^{d/2}|\Sigma|^{1/2}}\exp \left(-\frac{1}{2}(x-\mu _0)^T\Sigma^{-1}(x-\mu _0)\right) \\ p(x|y=1) & = & \frac{1}{(2\pi)^{d/2}|\Sigma|^{1/2}}\exp \left(-\frac{1}{2}(x-\mu _1)^T\Sigma^{-1}(x-\mu _1)\right)\end{align*}$$

Here, the parameters of our model are $\phi, \Sigma, \mu _0$ and $\mu _1$. The log-likelihood of the data is given by $$\begin{align*} l(\phi,\mu _0,\mu _1,\Sigma) & = & \log\prod _{i=1}^np\left(x^{(i)},y^{(i)};\phi,\mu _0,\mu _1, \Sigma\right) \\ & = & \log\prod _{i=1}^n p\left(x^{(i)}|y^{(i)};\mu _0, \mu _1, \Sigma\right)p\left(y^{(i)};\phi\right).\end{align*}$$

By maximizing $l$ with respect to the parameters, we find the maximum likelihood estimate of the parameters to be: $$\begin{align*}\phi & = & \frac{1}{n}\sum _{i=1}^n 1 \left\lbrace y^{(i)}=1\right\rbrace \\ \mu _0 & = & \frac{\sum _{i=1}^n 1\left\lbrace y^{(i)}=0\right\rbrace x^{(i)}}{\sum _{i=1}^n 1\left\lbrace y^{(i)}=0\right\rbrace} \\ \mu _1 & = & \frac{\sum _{i=1}^n 1\left\lbrace y^{(i)}=1\right\rbrace x^{(i)}}{\sum _{i=1}^n 1\left\lbrace y^{(i)}=1\right\rbrace} \\ \Sigma & = & \frac{1}{n} \sum _{i=1}^n\left(x^{(i)}-\mu _{y^{(i)}}\right)\left(x^{(i)}-\mu _{y^{(i)}}\right)^T.\end{align*}$$

With those idea, now I will demonstrate how this model works by solving Titanic survival prediction problem (first appears in [Titanic-survival-prediction](https://github.com/LeatuyrBertyk/Learning-ML/tree/main/2-Logistic_Regression)).

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ===== Preprocessing =====

# Load dataset
df = pd.read_csv('titanic_modified_dataset.csv', index_col = 'PassengerId') # Choose PassengerID as index

df = df.to_numpy().astype(np.float64)

X = df[:, :-1]
y = df[:, -1]

# Split Train, Validation, and Test
test_size = 0.3
random_state = 2
is_shuffle = True

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = test_size,
    random_state = random_state,
    shuffle = is_shuffle
)

(m, n) = X_train.shape

# Scaling data
normalizer = StandardScaler()
X_train[:, 1:] = normalizer.fit_transform(X_train[:, 1:])
X_test[:, 1:] = normalizer.transform(X_test[:, 1:])

In [7]:
# ===== Gaussian Discriminative Analysis =====
# Important functions
def prob_x_when_y(x, n, sigma, mu):
    det_sigma = max(np.linalg.det(sigma), 1e-10)
    a = np.sqrt((2*np.pi)**n * det_sigma)
    try:
        inv_sigma = np.linalg.inv(sigma)
        diff = x - mu
        exponent = -0.5 * diff.T @ inv_sigma @ diff
        return (1/a) * np.exp(exponent)
    except:
        return 1e-10

def prob_bernoulli(y, phi):
    return phi**y * (1 - phi)**(1 - y)

def predict(x, n, sigma, mu0, mu1, phi):
    prob_x_when_y0 = prob_x_when_y(x, n, sigma, mu0)
    prob_x_when_y1 = prob_x_when_y(x, n, sigma, mu1)
    prob_y0 = prob_bernoulli(0, phi)
    prob_y1 = prob_bernoulli(1, phi)

    prob_0 = prob_x_when_y0 * prob_y0
    prob_1 = prob_x_when_y1 * prob_y1

    return 1 if prob_1 > prob_0 else 0

# MLE for GDA
count_y0 = np.count_nonzero(y_train == 0)
count_y1 = np.count_nonzero(y_train == 1)

phi = count_y1 / m
mu0 = np.sum(X_train[y_train == 0], axis = 0) / count_y0
mu1 = np.sum(X_train[y_train == 1], axis = 0) / count_y1

sigma = np.zeros((n, n))
for i in range(m):
    mu = mu0 if y_train[i] == 0 else mu1
    diff = (X_train[i] - mu).reshape(-1, 1)
    sigma = sigma + diff @ diff.T
sigma = sigma/m

#print(f"m: {m}, n: {n}")
#print(f"phi: {phi}")
#print(f"mu0: {mu0}")
#print(f"mu1: {mu1}")
#print(f"sigma: {sigma}")

# Evaluate
gda_acc = sum([predict(X_test[i], n, sigma, mu0, mu1, phi) == y_test[i] for i in range(len(X_test))]) / len(X_test)

In [8]:
# ===== Mini-batch Logistic Regression =====
# Important functions
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def predict(theta, xi):
    z = np.dot(xi, theta)
    return sigmoid(z)

def compute_loss(yi, prediction):
    prediction = np.clip(prediction, 1e-15, 1 - 1e-15)
    return -(yi * np.log(prediction) + (1 - yi) * np.log(1 - prediction))

def compute_accuracy(y_true, y_predict_prob, threshold = 0.5):
    y_pred_class = (y_predict_prob >= threshold).astype(int)
    accuracy = np.mean(y_pred_class == y_true)
    return accuracy

# Mini-batch logistic regression
def minibatch_logistic_regression(X_train, y_train, batch_size = 64, 
                                   epochs = 10000, learning_rate = 1e-5):
    (m, n) = X_train.shape
    theta = np.zeros(n)
    
    for epoch in range(epochs):
        indecies = np.random.permutation(m)
        X_shuffled = X_train[indecies]
        y_shuffled = y_train[indecies]

        for i in range(0, m, batch_size):
            X_batch = X_shuffled[i: i + batch_size]
            y_batch = y_shuffled[i: i + batch_size]

            y_pred = predict(theta, X_batch)
            error = y_batch - y_pred
            theta = theta + (learning_rate / len(y_batch)) * (X_batch.T @ error)
    return theta

# Evaluate
theta_logistic = minibatch_logistic_regression(X_train, y_train, 32, 10000, 1e-5)
logistic_acc = compute_accuracy(y_test, predict(theta_logistic, X_test))

In [9]:
# ===== Scikit-learn model =====
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

# Remove bias
X_train_sk = X_train[:, 1:]
X_test_sk = X_test[:, 1:]

# Train model
model = LogisticRegression()
model.fit(X_train_sk, y_train)

# Evaluate
test_acc = accuracy_score(y_test, model.predict(X_test_sk))

In [10]:
# ===== Comparison =====
print(f"Gaussian Discriminant Analysis accuracy: {gda_acc * 100:.4f} %")
print(f"Mini-batch Losgistic Regression accuracy: {logistic_acc * 100:.4f} %")
print(f"Classification model Scikit-learn accuracy: {test_acc * 100:.4f} %")

Gaussian Discriminant Analysis accuracy: 76.8657 %
Mini-batch Losgistic Regression accuracy: 72.3881 %
Classification model Scikit-learn accuracy: 77.2388 %
